In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

elevation = dataset.select('DEM')

elev_projection = elevation.first().projection()

# 100m resolution
basins = ee.Image("projects/deforestation-495419/assets/hydrographic_basins").clip(panama_geom)

basins_int = basins.toInt()
neighbours = basins_int.focal_mode(1)

edges = basins_int.neq(neighbours).selfMask()

dist_basin = (
    edges.fastDistanceTransform(512).sqrt()
    .multiply(basins.projection().nominalScale())
    .clip(panama_geom)
)

### Original reprojection method

In [ ]:
# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_distance_basin = (
    dist_basin.resample("bilinear")
    .reproject(collection_projection)
    .rename("distance_basin_reprojected")
    .clip(panama_geom)
)

reprojected_distance_basin = dist_basin.reproject(
    crs = elevation.projection(), scale = dist_basin.projection().nominalScale()
)

# Map.addLayer(basins, {}, "Basins")
# Map.addLayer(edges, {"palette": ["red"]}, "Basin Edges")
Map.addLayer(dist_basin, {"min": 0, "max": 50000}, "Dist Basin")
Map.addLayer(reprojected_distance_basin, {"min": 0, "max": 50000}, "Dist Basin Reproj")

Map 

### Reproject by exporting

In [2]:
# export this as an image asset with the elev_projection and scale.
task = ee.batch.Export.image.toAsset(
    image = dist_basin,
    description = 'distance_hydro_basins',
    assetId = "projects/deforestation-panama/assets/distance_hydro_basins",
    region = panama_geom,
    scale = elev_projection.nominalScale(),
    crs = elev_projection.crs(),
    crsTransform = elev_projection.getInfo()['transform'],
    maxPixels=1e12
)

task.start()

In [ ]:
Map = geemap.Map()
Map.centerObject(panama_geom, 7)

export = ee.Image("projects/deforestation-panama/assets/distance_hydro_basins")

Map.addLayer(dist_basin, {}, 'Dist basin')
Map.addLayer(export, {}, "Dist basin reproj")

Map